In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import json
from PIL import Image
import os

In [2]:
#!unzip super-ai-engineer-ss-6-thai-language-image-captioning.zip

# preprocessing data

In [3]:
#create validation caption
with open("capgen_v1.0_val.json", "r", encoding="utf-8") as file:
    data_val = json.load(file)

# Convert JSON to a list of tuples (image_path, index, caption)
records = []
for image_path, captions in data_val.items():
    for idx, caption in enumerate(captions):
        records.append((image_path, idx, caption))

# Create DataFrame
caption_val = pd.DataFrame(records, columns=["image_path", "caption_index", "caption_text"])

In [4]:
#create training caption
with open("capgen_v1.0_train.json", "r", encoding="utf-8") as file:
    data_train = json.load(file)

# Convert JSON to a list of tuples (image_path, index, caption)
records = []
for image_path, captions in data_train.items():
    for idx, caption in enumerate(captions):
        records.append((image_path, idx, caption))

# Create DataFrame
caption_train = pd.DataFrame(records, columns=["image_path", "caption_index", "caption_text"])

In [5]:
#img = Image.open(image_path)
caption_train['from'] = caption_train["image_path"].apply(lambda x: x.split('/')[0])
caption_train['img_id'] = caption_train["image_path"].apply(lambda x: x.split('/')[-1])

In [6]:
caption_train

,image_path,caption_index,caption_text,from,img_id
0,coco/train2017/000000373716.jpg,0,ผู้หญิงสวมเสื้อแขนยาวสีขาวและเด็กนั่งเล่นกับสุ...,coco,000000373716.jpg
1,coco/train2017/000000373716.jpg,1,สาวคนนึงกำลังพาเด็กมานั่งเล่นอยู่ภายในสนามหญ้า...,coco,000000373716.jpg
2,coco/train2017/000000373716.jpg,2,ภาพขาวดำ ผู้หญิงนั่งบนพื้นอุ้มเด็กบนตัก ข้าง ๆ...,coco,000000373716.jpg
3,coco/train2017/000000196888.jpg,0,สีน้ำตาลตัวเล็กกำลังกินอาหารอยู่บนจานกระดาษสีข...,coco,000000196888.jpg
4,coco/train2017/000000196888.jpg,1,นกน้อยตัวหนึ่งกำลังจิกกินเศษอาหารที่วางทิ้งไว้...,coco,000000196888.jpg
...,...,...,...,...,...
430280,ipu24/train/food/28002.jpg,1,ขนมสับปะรดที่ใส่หมูสับใส่ในจานสีขาวและมีผักสวน...,ipu24,28002.jpg
430281,ipu24/train/food/28002.jpg,2,ในจานสีขาวมีสับปะรดหั่นเป็นชิ้นสามเหลี่ยมเล็ก ...,ipu24,28002.jpg
430282,ipu24/train/food/28003.jpg,0,ปลาดุกย่างเสียบไม้หลายตัวตั้งอยู่ในถาดมีผักแช่...,ipu24,28003.jpg
430283,ipu24/train/food/28003.jpg,1,ปลาดุกปิ้งจำนวนมากที่วางอยู่ในถาดอะลูมิเนียมทร...,ipu24,28003.jpg


In [7]:
caption_train_sub = caption_train[caption_train['caption_index']==0]
caption_train_sub = caption_train[caption_train['from']=='ipu24']

In [8]:
caption_train_sub = caption_train_sub.drop('caption_index', axis=1)
caption

'ปลาดุกย่างหลายตัวเสียบไม้วางอยู่บนตะแกรงที่มีถาดรองและด้านหน้ามีกะละมังที่แช่สะเดา'

In [9]:
caption_train_sub['from'].value_counts()

from
ipu24    85241
Name: count, dtype: int64

In [10]:
import os
import pandas as pd
from tqdm import tqdm  # Import tqdm for progress bar

# Function to load a single image based on row info
def load_image(row):
    # Define the base path for each source type
    if row['from'] == 'ipu24':
        if row['image_path'].split('/')[2] == 'travel':
            img_path = os.path.join('train/train/travel', row['img_id'])
        elif row['image_path'].split('/')[2] == 'food':
            img_path = os.path.join('train/train/food', row['img_id'])
    #elif row['from'] == 'coco':
        #img_path = os.path.join('/kaggle/input/coco-2017-dataset/coco2017/train2017', row['img_id'])

    return img_path


# Function to load images in batches and merge them into the same dataframe with a progress bar for the entire dataset
def load_images_with_progress(df, batch_size=100):
    all_batches = []  # List to store all processed batches
    
    # Create a progress bar for the entire dataset
    with tqdm(total=len(df), desc="Loading Images", unit="image") as pbar:
        for start in range(0, len(df), batch_size):
            batch = df.iloc[start:start + batch_size].copy()  # Create a copy of the batch to avoid SettingWithCopyWarning
            batch.loc[:, "new_image_path"] = batch.apply(load_image, axis=1)  # Load images for the batch
            all_batches.append(batch)  # Append the batch to the list
            
            # Update the progress bar after each batch
            pbar.update(batch_size)
    
    # Concatenate all batches back into a single dataframe
    df = pd.concat(all_batches, ignore_index=True)
    return df

In [11]:
train_real = load_images_with_progress(caption_train_sub, batch_size = 100)

Loading Images: 85300image [00:00, 101615.64image/s]                        


In [12]:
train_real.to_csv("train_real_with_path.csv")

In [13]:
train_real.head()

,image_path,caption_text,from,img_id,new_image_path
0,ipu24/train/travel/00000.jpg,รูปปั้นพญานาคตัวสีทองมีดอกบัวอยู่ด้านข้างอยู่ใ...,ipu24,00000.jpg,train/train/travel/00000.jpg
1,ipu24/train/travel/00000.jpg,รูปปั้นมังกรที่อยู่ภายในศาลเจ้าจีนและมีผ้าติดต...,ipu24,00000.jpg,train/train/travel/00000.jpg
2,ipu24/train/travel/00000.jpg,รูปปั้นมังกร มีหนวดสีทอง เขี้ยวและฟันสีขาว ลำต...,ipu24,00000.jpg,train/train/travel/00000.jpg
3,ipu24/train/travel/00001.jpg,โบราณสถานที่มีเจดีย์สีน้ำตาลดำมีต้นไม้หลายต้นด...,ipu24,00001.jpg,train/train/travel/00001.jpg
4,ipu24/train/travel/00001.jpg,สร้างโบราณสถานขนาดใหญ่ที่อยู่ตรงพื้นที่เต็มไปด...,ipu24,00001.jpg,train/train/travel/00001.jpg


In [14]:
train_real_fr = train_real.drop(['image_path', 'from', 'img_id',], axis=1)
train_real_fr

,caption_text,new_image_path
0,รูปปั้นพญานาคตัวสีทองมีดอกบัวอยู่ด้านข้างอยู่ใ...,train/train/travel/00000.jpg
1,รูปปั้นมังกรที่อยู่ภายในศาลเจ้าจีนและมีผ้าติดต...,train/train/travel/00000.jpg
2,รูปปั้นมังกร มีหนวดสีทอง เขี้ยวและฟันสีขาว ลำต...,train/train/travel/00000.jpg
3,โบราณสถานที่มีเจดีย์สีน้ำตาลดำมีต้นไม้หลายต้นด...,train/train/travel/00001.jpg
4,สร้างโบราณสถานขนาดใหญ่ที่อยู่ตรงพื้นที่เต็มไปด...,train/train/travel/00001.jpg
...,...,...
85236,ขนมสับปะรดที่ใส่หมูสับใส่ในจานสีขาวและมีผักสวน...,train/train/food/28002.jpg
85237,ในจานสีขาวมีสับปะรดหั่นเป็นชิ้นสามเหลี่ยมเล็ก ...,train/train/food/28002.jpg
85238,ปลาดุกย่างเสียบไม้หลายตัวตั้งอยู่ในถาดมีผักแช่...,train/train/food/28003.jpg
85239,ปลาดุกปิ้งจำนวนมากที่วางอยู่ในถาดอะลูมิเนียมทร...,train/train/food/28003.jpg


# Train the model

In [15]:
instruction = "คิดแคปชั่นรูปนี้ให้หน่อย"
def convert_to_conversation(sample):
    conversation = [
        { "role": "user",
          "content" : [
            {"type" : "text",  "text"  : instruction},
            {"type" : "image", "image" : Image.open(sample["new_image_path"])} ]
        },
        { "role" : "assistant",
          "content" : [
            {"type" : "text",  "text"  : sample["caption_text"]} ]
        },
    ]
    return { "messages" : conversation }

In [16]:
converted_dataset = []
for index, row in tqdm(train_real_fr.iterrows()):
    converted_dataset.append(convert_to_conversation(row))
#converted_dataset = [convert_to_conversation(sample) for sample in train_real]

85241it [00:10, 7891.52it/s] 


# real training

In [17]:
%%bash
pip install unsloth
# Also get the latest nightly if needed:
pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

  Using cached unsloth-2026.4.2-py3-none-any.whl.metadata (55 kB)
  Using cached unsloth_zoo-2026.4.2-py3-none-any.whl.metadata (32 kB)
  Using cached torch-2.10.0-3-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached torchvision-0.26.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached tyro-1.0.12-py3-none-any.whl.metadata (12 kB)
  Using cached xformers-0.0.35-py39-none-manylinux_2_28_x86_64.whl.metadata (1.2 kB)
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (1.7 kB)
  Using cached sentencepiece-0.2.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (10 kB)
  Using cached datasets-4.3.0-py3-none-any.whl.metadata (18 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached peft-0.18.1-py3-none-any.whl.metadata (14 kB)
  Using cached huggingface_hub-1.9.0-py3-none-

Found existing installation: unsloth 2026.4.2
Uninstalling unsloth-2026.4.2:
  Successfully uninstalled unsloth-2026.4.2


  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-req-build-qe570dy5


  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-req-build-qe570dy5


  Resolved https://github.com/unslothai/unsloth.git to commit 610086744766a052cb799b17538b1238c7931659
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for unsloth: filename=unsloth-2026.4.2-py3-none-any.whl size=29540575 sha256=e37055a3ae8fc3932c56936f0d3891928b0c9185a08abce482678fc7ef30be8c
  Stored in directory: /tmp/pip-ephem-wheel-cache-3hzq_vx0/wheels/60/3e/1f/e576c07051d90cf64b6a41434d87ccf4db33fafd5343bf5de0
Successfully built unsloth


In [18]:
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-4B",
    load_in_4bit = False, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.1: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [19]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

Unsloth: Making `model.base_model.model.model.visual` require gradients


In [20]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3_5ForConditionalGeneration(
      (model): Qwen3_5Model(
        (visual): Qwen3_5VisionModel(
          (patch_embed): Qwen3_5VisionPatchEmbed(
            (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
          )
          (pos_embed): Embedding(2304, 1024)
          (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
          (blocks): ModuleList(
            (0-23): 24 x Qwen3_5VisionBlock(
              (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
              (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
              (attn): Qwen3_5VisionAttention(
                (qkv): lora.Linear(
                  (base_layer): Linear(in_features=1024, out_features=3072, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Identity()
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_

In [21]:
from unsloth import is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model) # Enable for training!

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer), # Must use!
    train_dataset = converted_dataset,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 150,
        # num_train_epochs = 1, # Set this instead of max_steps for full training runs
        learning_rate = 2e-4,
        fp16 = not is_bf16_supported(),
        bf16 = is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",     # For Weights and Biases

        # You MUST put the below items for vision finetuning:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        dataset_num_proc = 4,
        max_seq_length = 2048,
    ),
)

Unsloth: Model does not have a default image size - using 512


[accelerate.utils.other|WARNING][RANK 0] Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [22]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 85,241 | Num Epochs = 1 | Total steps = 150
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 38,756,352 of 4,578,021,888 (0.85% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,3.139397
2,3.083565
3,2.873024
4,2.657189
5,2.501003
6,2.171138
7,1.568380
8,1.699782
9,1.532773
10,1.508646


# Process the test set

In [49]:
FastVisionModel.for_inference(model) # Enable for inference!

instruction = "คิดแคปชั่นรูปนี้ให้หน่อย"
test_cap = []
def generateCaption(image_path):
    image = Image.open(image_path)
    messages = [
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": instruction}
        ]}
    ]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
    inputs = tokenizer(
        image,
        input_text,
        add_special_tokens = False,
        return_tensors = "pt",
    ).to("cuda")
    
    output = model.generate(**inputs, max_new_tokens = 100,
                       use_cache = True, temperature = 0.7, min_p = 0.1)
    caption = tokenizer.decode(output[0], skip_special_tokens=True)
    test_cap.append(caption)

In [50]:
folder_path = 'test/test'

image_paths = []
for root, dirs, files in os.walk(folder_path):
    for file in files:
            image_paths.append(os.path.join(root, file))

print(len(image_paths))


2000


In [ ]:
for img_path in tqdm(image_paths):
    generateCaption(img_path)

  1%|▏         | 25/2000 [01:05<1:12:51,  2.21s/it]

In [ ]:
test_cap[0]

In [ ]:
image_id = [k.split('/')[-1] for k in image_paths]

In [ ]:
image_id_real = [k.split('.')[0] for k in image_id]

In [ ]:
data = {'image_id': image_id_real, 'cc': test_cap}

# Create a DataFrame
submission = pd.DataFrame(data)

In [ ]:
submission.to_csv("test_ans_1.csv")

In [ ]:
submission['cc'] = submission['cc'].apply(lambda x: x.split('\n')[-1])


In [ ]:
submission['cc'] = submission['cc'].apply(lambda x: x.replace('"',''))

In [ ]:
submission

In [ ]:
submission.to_csv("first_sub1.csv")

# align labels

In [ ]:
sample = pd.read_csv('sample_submission.csv', dtype={'image_id': str})

In [ ]:
sampleid = sample['image_id']


In [ ]:
id_list = sampleid.to_list()

In [ ]:
id_list[:5]

In [ ]:
submission.head()

In [ ]:
df_final = submission.set_index('image_id').loc[id_list].reset_index()

In [ ]:
df_final = df_final.rename(columns={'cc': 'caption'})

In [ ]:
df_final['caption'][0]

In [44]:
df_final

,image_id,caption
0,01354,ราสีขาวที่ขึ้นอยู่บริเวณบนใบไม้สีน้ำตาลขนาดใหญ่
1,01413,งูเห่าตัวหนึ่งที่มีสีน้ำตาลและสีเทาอยู่บนพื้นไม้
2,01802,บ้านไม้หลายหลังที่ตั้งอยู่บนหน้าผาหินขนาดใหญ่แ...
3,01243,ลิงตัวหนึ่งนั่งอยู่ใกล้กับกำแพงสีขาวและมีหญ้าอ...
4,00693,แมลงที่มีปีกสีน้ำตาลมีจุดสีขาวและจุดสีส้มอยู่บ...
...,...,...
1995,00029,ภายในอาคารมีต้นไม้หลายต้น มีบ่อน้ำ มีป้าย มีคน...
1996,01065,ดอกสีแดงหลายดอกติดอยู่บนกิ่งไม้และมีใบสีเขียวอ...
1997,00195,ต้นไม้ขนาดใหญ่ที่มีใบสีเขียวตั้งอยู่บริเวณริมถ...
1998,01026,ดอกสีเหลืองและสีน้ำตาลติดอยู่บนต้นไม้อยู่ในป่า


In [ ]:
df_final.to_csv('QWEN3.5_caption_temp0.7.csv', index=False)